In [ ]:
# Importando as bibliotecas

import sys
!{sys.executable} -m pip install torch torchvision torchaudio
import torch
import torch.nn as nn
import torch.nn.functional as F 
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import numpy as np
import pickle


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Windows\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable


In [ ]:
from torch.utils.data import Dataset
import numpy as np

# Função fornecida no site do Dataset
def unpickle(file):
    import pickle
    with open(file, 'rb') as fo:
        dicionario = pickle.load(fo, encoding='bytes')
    return dicionario

class DatasetCIFAR(Dataset):
    def __init__(self, file_path, transform=None):
        # Carrega o dicionário
        dicionario = unpickle(file_path)
        
        # Guarda as imagens na chave data e os rótulos em labels
        self.dados = dicionario[b'data']
        self.rotulos = dicionario[b'labels']
        self.transform = transform
        
        # N = número de imagens, 3 = canais RGB, 32x32 = tamanho da imagem
        self.dados = self.dados.reshape(-1, 3, 32, 32)

    def __len__(self):
        # Retorna a quantidade total de imagens no arquivo
        return len(self.dados)

    def __getitem__(self, idx):
        # Pega uma imagem e seu rótulo específico
        imagem = self.dados[idx]
        rotulo = self.rotulos[idx]
        
        # Imagens em array no formato (Altura, Largura, Canais) para aplicar as transformações do PyTorch
        # Trocamos a ordem de (3, 32, 32) para (32, 32, 3) temporariamente para aplicar as transformações 
        imagem = np.transpose(imagem, (1, 2, 0))
        
        if self.transform:
            imagem = self.transform(imagem)
            
        return imagem, rotulo

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # in_channels=3 (RGB) e out_channels=16 (quantidade de filtros) 
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1) # Convolução 1
        self.pool = nn.MaxPool2d(2, 2) # Max Pooling 
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1) # Convolução 2
        
        # Após 2 pools, 32x32 vira 8x8
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    # A função forward define como os dados fluem pela rede
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32 * 8 * 8) 
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Cria o modelo
modelo = CNN()

# Configura o transformer para normalizar as imagens
transformacao = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Carrega os dados (lembre de colocar o caminho certo do seu arquivo!)
caminho_do_arquivo = 'cifar-10-batches-py/test_batch' 
dataset_teste = DatasetCIFAR(caminho_do_arquivo, transform=transformacao)
dataloader_teste = DataLoader(dataset_teste, batch_size=32, shuffle=False)

C:\Users\Windows\AppData\Local\Temp\ipykernel_24368\2980694562.py:8: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  dicionario = pickle.load(fo, encoding='bytes')


In [ ]:
import torch.optim as optim
import torchvision

# Carrega o dataset de TREINO (train=True)
dataset_treino = torchvision.datasets.CIFAR10(root='./dados', train=True, download=True, transform=transformacao)

# shuffle=True embaralha as imagens e a rede não decora a ordem
dataloader_treino = DataLoader(dataset_treino, batch_size=32, shuffle=True)

# Calcula a entropia cruzada (O quão perto as previsões estão dos rótulos reais)
criterio = nn.CrossEntropyLoss()

# O Adam vai ajustar os neurônios da rede para diminuir o erro na próxima tentativa
otimizador = optim.Adam(modelo.parameters(), lr=0.001) # lr = learning rate

100.0%
C:\Users\Windows\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [ ]:
epocas = 5 # 5 épocas de treino

print("Iniciando o treinamento...")

for epoca in range(epocas):
    modelo.train() 
    perda_acumulada = 0.0
    
    for imagens, rotulos in dataloader_treino:
        
        # Limpar a memória do aprendizado anterior
        otimizador.zero_grad()
        
        # Forward (A rede tenta adivinhar o que é a imagem)
        saidas = modelo(imagens)
        
        # Calcular o erro 
        erro = criterio(saidas, rotulos)
        
        # Backward 
        erro.backward()
        
        # Atualiza os pesos
        otimizador.step()
        
        perda_acumulada += erro.item()
        
    # Mostra o resultado ao final de cada época
    erro_medio = perda_acumulada / len(dataloader_treino)
    print(f"Época {epoca+1}/{epocas} concluída! Erro médio: {erro_medio:.4f}")

print("Treinamento finalizado!")

Iniciando o treinamento...
Época 1/5 concluída! Erro médio: 1.3543
Época 2/5 concluída! Erro médio: 1.0123
Época 3/5 concluída! Erro médio: 0.8608
Época 4/5 concluída! Erro médio: 0.7526
Época 5/5 concluída! Erro médio: 0.6595
Treinamento finalizado!


In [ ]:
import torch

modelo.eval() 
acertos = 0
total = 0

# Desliga o cálculo de gradiente para economizar memória e acelerar a inferência
with torch.no_grad():
    for imagens, rotulos in dataloader_teste:
        saidas = modelo(imagens)
        _, previsoes = torch.max(saidas, 1)
        
        total += rotulos.size(0)
        acertos += (previsoes == rotulos).sum().item()

print(f'Acurácia: {100 * acertos / total:.2f}%')

Acurácia: 69.07%
